# 12 · Combining concurrency mechanisms

**Time:** 35–45 minutes. **You need:** lessons 02–05 and 09–11;
Python 3.11+. The runnable example uses Python's standard library.

You will combine file-reading threads, CPU processes, and async coordination.
You will limit submissions as well as queued results, preserve sample IDs,
and stop the pipeline when one input is corrupt.

Everything stays local. The example creates small temporary binary files,
performs synthetic byte transforms, and uses a slow consumer simulation.
It does not decode images or run a model or GPU workload.

[Course guide](README.md) · Previous: [Native library parallelism](11_native_library_parallelism.ipynb)
· Next: [Advanced DataLoader](13_advanced_dataloader.ipynb)

## 1. Assign a job to each mechanism

| Stage | Mechanism | Purpose |
| --- | --- | --- |
| Read a local file | Thread via `asyncio.to_thread` | Keep blocking file I/O away from the event loop |
| Transform its bytes | `ProcessPoolExecutor` | Run a Python CPU loop in separate processes |
| Coordinate stages and queue | `asyncio` tasks | Pause while another stage works or a queue is full |
| Consume small batches | Main event loop | Produce one feature per sample; a sleep models downstream waiting |

The event loop is the coordinator, not the CPU worker. Threads read the files;
processes transform the bytes. [Sending blocking calls to threads](https://docs.python.org/3.11/library/asyncio-task.html#asyncio.to_thread)
and [using a process executor from the event loop](https://docs.python.org/3.11/library/asyncio-eventloop.html#asyncio.loop.run_in_executor).

A **coordinator** is an async task that moves one sample through the stages. A **payload** is the sample's byte data. The sample ID and file path are **metadata**: small descriptions of that data. **KiB** means 1,024 bytes.

```mermaid
flowchart LR
  A[Iterator of sample IDs and file paths] --> B[Two async coordinators]
  B --> C[Thread reads one file]
  C --> D[Process transforms its bytes]
  D --> E[Bounded result queue: two items]
  E --> F[One main-loop consumer: batches of two]
  E -. Full queue pauses coordinator .-> B
```

Each coordinator repeats: claim one path → await its read → await its
transformation → await room in the result queue. Only then can it claim another
path. Two coordinators can overlap those steps for two different samples.

If your notebook viewer does not render Mermaid, follow that same sequence
from the path iterator to the consumer.

In [1]:
import inspect
import json
import subprocess
import sys
from pathlib import Path
from zlib import crc32

if sys.version_info < (3, 11):
    raise RuntimeError("Use Python 3.11 or later for this lesson.")
PROJECT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "lessons" / "pipeline_demo.py").exists()),
    None,
)
if PROJECT is None:
    raise FileNotFoundError("Open this notebook from the course folder.")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from lessons import pipeline_demo as demo
HELPER = PROJECT / "lessons" / "pipeline_demo.py"


def run_pipeline_demo(*arguments):
    completed = subprocess.run(
        [sys.executable, "-X", "utf8", str(HELPER), *map(str, arguments)],
        cwd=PROJECT, capture_output=True, text=True, encoding="utf-8", timeout=90,
    )
    if completed.returncode:
        raise RuntimeError(
            f"Helper exited with code {completed.returncode}:\n"
            + completed.stdout + "\n" + completed.stderr
        )
    if completed.stderr:
        print(completed.stderr)
    return json.loads(completed.stdout)


print("Helper:", HELPER.name)

Helper: pipeline_demo.py


The helper uses a guarded entry point and explicitly selects `spawn` so the
process example works on Windows as well as other desktop systems. It uses
`asyncio.run()` in its separate interpreter. Jupyter already has an event loop;
launching the helper lets it own its loop and its worker cleanup.
[Spawn requirements](https://docs.python.org/3.11/library/multiprocessing.html#the-spawn-and-forkserver-start-methods).

## 2. Define the input and the output

Each file contains a short identifying header and 32 KiB of generated bytes.
`MAGIC = b"MLDEMO1\n"` is the header used by this lesson; `b"..."` means
a sequence of bytes. Checking it catches our deliberately damaged file,
although a valid header alone does not prove all the file contents are intact.
`read_file` really reads the file. Its optional delay models slower storage
and defaults to zero. `transform` checks the header and performs three passes
of Python arithmetic over those bytes.

The consumer produces one compact integer fingerprint per sample using `zlib.crc32`.
The fingerprint depends on byte content and order. A simple sum would miss the
reordering in these generated examples. The sample ID travels with the bytes, so
we can check for duplicate or missing IDs even when completion order changes.

A fingerprint is a useful small check, but two different payloads can have the
same fingerprint. Comparing full arrays is stronger when keeping them is affordable.
[Python CRC32 reference](https://docs.python.org/3/library/zlib.html#zlib.crc32).

In [2]:
print(inspect.getsource(demo.read_file))
print(inspect.getsource(demo.transform))

# Show why a byte sum would be a weak check for this transformation.
example_bytes = bytes(range(256)) * 4
_, example_transformed = demo.transform((0, demo.MAGIC + example_bytes), rounds=3)
print("Byte sums equal:", sum(example_bytes) == sum(example_transformed))
print("Fingerprints equal:", crc32(example_bytes) == crc32(example_transformed))
assert sum(example_bytes) == sum(example_transformed)
assert crc32(example_bytes) != crc32(example_transformed)

def read_file(item, read_delay):
    job_id, path = item
    data = path.read_bytes()  # Actual blocking file I/O occurs in a thread.
    if read_delay:
        time.sleep(read_delay)  # Optional, explicit simulation of slower storage.
    return job_id, data

def transform(item, rounds):
    """Synthetic byte processing in Python. This is not an image decoder."""
    job_id, data = item
    if not data.startswith(MAGIC):
        raise ValueError(f"Corrupt input {job_id}: missing sample header")
    data = data[len(MAGIC):]
    for _ in range(rounds):
        data = bytes((value * 3 + job_id) % 256 for value in data)
    return job_id, data

Byte sums equal: True
Fingerprints equal: False


A regular sequential version reads, transforms, and consumes the same generated
inputs in the same-sized batches. It uses the same downstream delay. We compare
the final dictionary of IDs and features against that baseline.

## 3. Bound the submissions as well as the queue

A result queue limits **finished items waiting for the consumer**. It does not,
on its own, limit the number of executor jobs or async tasks you create.

A loop that submits a million files immediately can retain a million task
records, arguments, or future results before they reach a small queue. Our
design creates a fixed number of coordinator tasks instead.

With `C` coordinators:

- At most `C` file reads are awaiting completion.
- At most `C` process calls are awaiting completion, counting running and pending calls.
- Each coordinator owns at most one sample as it travels through the stages.
- At most `Q` items fit in a queue of capacity `Q`.
- The consumer retains at most `B` payloads while forming/using a batch.

The read and process bounds overlap: a coordinator awaits one stage before
submitting the next. We never create one coordinator per input.

In the helper excerpts, `args` holds the settings chosen on the command line, such as `--rounds 3`. `stats` is a dictionary of observed counts, used to check the limits. `raw` holds the bytes before transformation and `processed` holds the result. All counter updates happen on the single event-loop thread.

In [3]:
print(inspect.getsource(demo.coordinate))

async def coordinate(inputs, results, pool, args, stats):
    """One coordinator owns at most one job as it moves through the stages."""
    # args holds command-line settings; stats holds observed activity counts.
    loop = asyncio.get_running_loop()
    while True:
        # No await during this claim: other tasks on this event loop cannot
        # interrupt next(). The iterator contains paths, not loaded samples.
        item = next(inputs, None)
        if item is None:
            break
        stats["claimed"] += 1
        stats["active_reads"] += 1
        stats["peak_reads"] = max(stats["peak_reads"], stats["active_reads"])
        try:
            raw = await asyncio.to_thread(read_file, item, args.read_delay)
        finally:
            stats["active_reads"] -= 1

        stats["active_process_calls"] += 1
        stats["peak_process_calls"] = max(
            stats["peak_process_calls"], stats["active_process_calls"])
        try:
            processed = await loop.run_in

Read that function in four steps. `next(inputs, None)` claims a path.
`await asyncio.to_thread(...)` waits for the file-reading thread.
`await loop.run_in_executor(...)` waits for the CPU process.
`await results.put(...)` waits for queue space.

These tasks all claim paths on one event loop. There is no `await` inside the
claim, so two async tasks cannot interrupt that short iterator operation.
The threads and processes do not touch the iterator or its counters.

If the queue is full, `put()` pauses the coordinator. It keeps its finished
sample but submits no new read or process job. That pause is **backpressure**:
the slower consumer controls how far the upstream work can get ahead.
[Bounded asyncio queues](https://docs.python.org/3.11/library/asyncio-queue.html#asyncio.Queue).

## 4. Consume batches without keeping all payloads

The consumer keeps a small batch, records the integer features, and clears the
batch. It does not save all transformed byte arrays. Each successful coordinator
eventually sends `None` as its completion marker. Because its marker follows
its own results, seeing all markers means all successful inputs have passed
through the queue. A partial final batch is processed too.

Using a special value to mark completion is often called a **sentinel**. Here,
`None` is safe because a real result is always a `(job_id, data)` tuple.

In [4]:
print(inspect.getsource(demo.consume))

async def consume(results, coordinator_count, batch_size, consumer_delay, stats):
    """A single consumer lives on the main event loop; retain only small features."""
    finished = 0
    batch = []
    seen = {}

    async def use_batch():
        # This wait models a busy downstream consumer; there is no GPU work here.
        await asyncio.sleep(consumer_delay)
        for job_id, data in batch:
            if job_id in seen:
                raise AssertionError(f"Duplicate sample {job_id}")
            seen[job_id] = crc32(data)  # A compact fingerprint sensitive to byte order.
        stats["batches"] += 1
        batch.clear()

    while finished < coordinator_count:
        item = await results.get()
        if item is None:
            finished += 1
        else:
            batch.append(item)
            item = None
            stats["peak_batch"] = max(stats["peak_batch"], len(batch))
            if len(batch) == batch_size:
                await use_batch()
    if batch:
  

The sleep in `use_batch()` models a consumer that takes time to finish a batch.
It lets the event loop run other tasks while waiting. A real blocking consumer
would need an appropriate thread, process, or asynchronous interface. A long
Python CPU loop on this event loop would delay every coordinator.

**Predict:** will changing the result queue from one slot to four let the
consumer work faster? What changes if the consumer is the slowest stage?

In [5]:
normal = run_pipeline_demo()
assert normal["status"] == "success"
assert normal["all_results_match"] and normal["executors_closed"]
print(f"Sequential, same inputs/delays: {normal['sequential_seconds']:.4f} s")
print(f"Pipeline, including startup:   {normal['pipeline_seconds']:.4f} s")
print("Observed limits:", normal["stats"])

Sequential, same inputs/delays: 0.4920 s
Pipeline, including startup:   0.5541 s
Observed limits: {'claimed': 12, 'active_reads': 0, 'peak_reads': 2, 'active_process_calls': 0, 'peak_process_calls': 2, 'peak_queue': 2, 'peak_batch': 2, 'batches': 6, 'full_queue_observations': 8}


`peak_process_calls` counts submitted calls that a coordinator is awaiting,
including calls pending in the process executor. It is not a count of CPU cores
active at an instant. `full_queue_observations` counts times a producer found
the queue full; timing and scheduling can change this value between runs.

Startup can make this small pipeline slower than the sequential baseline.
The goal here is correct bounded coordination. A speedup requires enough useful
work and enough opportunity to overlap stages.

`peak_queue` counts occupied queue slots, which can include completion markers. The capacity limits all entries, even though a marker holds much less data than a sample.

In [6]:
observations = []
for capacity in (1, 4):
    report = run_pipeline_demo("--capacity", capacity, "--consumer-delay", 0.10)
    assert report["all_results_match"]
    assert report["stats"]["peak_queue"] <= capacity
    observations.append(report)
    print(f"capacity={capacity}: {report['pipeline_seconds']:.4f} s; "
          f"peak queue={report['stats']['peak_queue']}; "
          f"full observations={report['stats']['full_queue_observations']}")

capacity=1: 0.7732 s; peak queue=1; full observations=9


capacity=4: 0.7791 s; peak queue=4; full observations=6


More queue space can smooth short bursts. It cannot reduce the time the consumer
needs for each batch. A larger observed queue means more payloads held in memory.
Never conclude that a capacity is universally better from one short timing.

## 5. Estimate a memory budget carefully

The queue alone holds up to `Q × payload_bytes`. Add up to `C` samples owned
by coordinators and up to `B` samples owned by the consumer. That gives a useful
first count of logical payloads: `(Q + C + B) × payload_bytes`.

This is **not an exact memory limit**. Sending bytes to a process creates
serialization buffers and worker copies. A transformation can temporarily
retain both its input and output. Python objects, the interpreters, imported
modules, OS buffers, and file caching cost memory too.

The helper also stores all generated path metadata and final integer features,
which grow with the number of inputs. In a much larger application, read paths
from an iterator and write features incrementally. The sample payloads and
outstanding executor submissions remain bounded here.

In [7]:
C = normal["coordinators"]
Q = normal["queue_capacity"]
B = normal["batch_size"]
sample_bytes = normal["payload_bytes"]
print(f"Logical payload estimate: {(C + Q + B) * sample_bytes / 1024:.0f} KiB")
print("Plus copies, buffers, worker memory, metadata, and compact output features.")
assert normal["stats"]["peak_reads"] <= C
assert normal["stats"]["peak_process_calls"] <= C
assert normal["stats"]["peak_batch"] <= B

Logical payload estimate: 192 KiB
Plus copies, buffers, worker memory, metadata, and compact output features.


## 6. Make failure stop the group

On the success path, the consumer uses completion markers to stop. On the
failure path, a coordinator might never send its marker. Waiting for all markers
anyway would hang the consumer.

A `TaskGroup` keeps the consumer and coordinators in one group. If a child
raises an ordinary exception, the group cancels its other tasks and reports the
failure. That includes tasks paused on a full queue or an empty queue. The
helper does not wait for a queue drain after a failure.
[TaskGroup failure behavior](https://docs.python.org/3.11/library/asyncio-task.html#task-groups).

Here, **cancellation** asks an async task to stop at a point where it is awaiting. Related failures are returned in an **exception group**, a container holding one or more errors from those tasks.

In [8]:
print(inspect.getsource(demo.run_pipeline))

async def run_pipeline(items, pool, args, stats):
    results = asyncio.Queue(maxsize=args.capacity)
    inputs = iter(items)
    async with asyncio.TaskGroup() as group:
        consumer = group.create_task(consume(
            results, args.coordinators, args.batch_size, args.consumer_delay, stats))
        for _ in range(args.coordinators):
            group.create_task(coordinate(inputs, results, pool, args, stats))
    # TaskGroup waits for all tasks; a task failure cancels its siblings.
    return consumer.result()



Cancelling an awaiting coordinator does **not** stop a thread or process call
that is already running. `asyncio.run()` closes its default thread executor.
A `finally` block shuts down the process pool with `cancel_futures=True`:
queued work that has not started may be cancelled; running finite calls finish
before workers are joined.

Our jobs are small and finite, and workers never wait on the result queue.
That avoids a shutdown cycle where the parent waits for a worker that waits
for the cancelled consumer. Executor shutdown has no hard time limit if a
running function or operating-system call hangs. The notebook launcher has a
timeout to report such a failure, not a promise to kill every worker safely.
[Executor shutdown and cancellation](https://docs.python.org/3.11/library/concurrent.futures.html#concurrent.futures.Executor.shutdown).

**Predict:** corrupt sample 3, use a queue of one item, and slow the consumer.
Will the helper hang waiting for its missing completion marker?

In [9]:
failed = run_pipeline_demo("--fail-id", 3, "--capacity", 1,
                           "--consumer-delay", 0.10)
assert failed["status"] == "expected_failure"
assert failed["executors_closed"]
assert any("Corrupt input 3" in message for message in failed["errors"])
print("Detected:", failed["errors"])
print("Worker pools closed:", failed["executors_closed"])
print("No complete output is reported for a failed run.")

Detected: ['Corrupt input 3: missing sample header']
Worker pools closed: True
No complete output is reported for a failed run.


The helper recognizes the deliberately introduced `ValueError` for this
experiment. Unexpected errors still fail the subprocess and the notebook cell.
The error reaches the main process; it is never treated as a missing sample
in an otherwise successful result.

## 7. Completion order and correctness

Faster samples can reach the consumer first. This consumer calculates an
independent feature per ID, so arrival order does not matter. It checks for
duplicates, and its dictionary must equal the sequential dictionary.

If your consumer needs original order, keep the IDs and define an explicit
ordering policy. Holding all later results while waiting for one slow early
result can grow memory. Limit that reorder buffer as well as the queue and
submissions. Ordering is a requirement to design for, not an automatic benefit
of a queue.

## 8. Practice

1. With `--coordinators 2 --workers 4`, what limits the number of simultaneous
   process calls? Could all four workers stay busy?
2. Run 13 jobs with a batch size of 4. How many batches should the consumer use?
   Check that it processes the last partial batch.
3. Set `--read-delay 0.02` to simulate slower storage after each actual read.
   Compare one and two coordinators. Keep all other settings fixed.
4. Replace the fixed coordinator count with one task per file. Why would a
   bounded result queue alone no longer establish the same memory budget?
5. Why does cancelling an async task not guarantee immediate shutdown of its
   submitted CPU work?

In [10]:
# Verify the partial final batch without changing the helper.
partial = run_pipeline_demo("--jobs", 13, "--batch-size", 4)
assert partial["all_results_match"]
assert partial["stats"]["batches"] == 4
print("13 samples ->", partial["stats"]["batches"], "batches")

# Optional comparison: introduce an explicitly simulated storage delay.
# one = run_pipeline_demo("--coordinators", 1, "--read-delay", 0.02)
# two = run_pipeline_demo("--coordinators", 2, "--read-delay", 0.02)

13 samples -> 4 batches


<details>
<summary>Worked answers</summary>

1. The two coordinators can submit at most two process calls before awaiting
   their results. Four process workers cannot all stay busy in this design.
   Increasing a pool's worker count does not create more outstanding work.
2. Four batches: three full batches and a final batch with one sample. The
   completion markers end input, then the consumer handles its partial batch.
3. Two coordinators can overlap the artificial waits and some processing.
   A slow consumer or process startup can still dominate total time. Explain
   the actual measurements and identify the delay as simulated.
4. All those tasks might already hold payloads or submitted future results.
   They can accumulate before they try to put into the queue. Submission and
   task counts need their own limits.
5. The async task coordinates work; it does not own the executing process.
   Cancelling the wait may cancel work that has not started, but running
   executor calls normally finish. Cleanup must account for that distinction.

</details>

You can now explain how different stages use different concurrency tools,
how the bounds reach upstream, and how failures reach the main process.
Next: [Advanced DataLoader](13_advanced_dataloader.ipynb).